# Stage 1–2 FITS-PSF evaluation
Evaluates `train_HSC_PSF.py` weights using the empirical FITS PSF and `model.eval()`.

In [ ]:
import torch, torch.nn.functional as F, matplotlib.pyplot as plt
import data
from differentiable_lensing import DifferentiableLensing
from psf import build_psf_kernel, apply_psf
from sisr import SISR
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHTS='hsc_fits_psf_weights.pt'; PSF_PATH='path/hsc.fits'; RESOLUTION=0.168; PSF_NATIVE_SCALE=0.168; VAL_CLASS='no_sub'; VAL_INDEX=0
MAG=2; TARGET_SHAPE=128; TARGET_RESOLUTION=RESOLUTION/MAG

In [ ]:
model=SISR(MAG,1,3,in_channels=2,latent_channel_count=64).to(DEVICE)
state=torch.load(WEIGHTS,map_location=DEVICE); state=state.get('model_state_dict',state) if isinstance(state,dict) else state
model.load_state_dict(state); model.eval()
lens=DifferentiableLensing(device=DEVICE,alpha=None,target_resolution=TARGET_RESOLUTION,target_shape=TARGET_SHAPE).to(DEVICE)
backward=torch.load('sparse_grid_fracs_euclid_backward.pt',map_location=DEVICE).to(DEVICE)
forward=[torch.load('scatter_to_log_128.pt',map_location=DEVICE).to(DEVICE),torch.load('forward_from_log_128.pt',map_location=DEVICE).to(DEVICE),torch.load('scatter_from_log_128.pt',map_location=DEVICE).to(DEVICE)]
psf=build_psf_kernel('fits',0.16,TARGET_RESOLUTION,path=PSF_PATH,source_pixscale_arcsec=PSF_NATIVE_SCALE,device=DEVICE)
print('PSF',tuple(psf.shape),float(psf.sum()))

In [ ]:
ds=data.LensingDataset('val/',[VAL_CLASS],VAL_INDEX+1); lr=ds[VAL_INDEX].unsqueeze(0).float().to(DEVICE)
if lr.ndim==5: lr=lr.squeeze(1)
with torch.no_grad():
    source_lr=lens.cross_grid_fill(lr,[backward])
    source_hr=model(torch.cat([source_lr,lr],dim=1))
    intrinsic=lens.cross_grid_fill(source_hr,forward)
    convolved=apply_psf(intrinsic,psf)
    pred=F.interpolate(convolved,size=lr.shape[-2:],mode='area')
zero=lr.square().mean(); mse=F.mse_loss(pred,lr); print({'zero_mse':float(zero),'model_mse':float(mse),'skill_over_zero':float(1-mse/zero.clamp_min(1e-8))})
fig,ax=plt.subplots(1,3,figsize=(12,4))
for a,x,t in zip(ax,[lr[0,0],pred[0,0],pred[0,0]-lr[0,0]],['LR observation','FITS-PSF prediction','Residual']): a.imshow(x.cpu(),cmap='gray'); a.set_title(t)
plt.tight_layout()